In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from sklearn.metrics import roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
import os
from tqdm import tqdm
import random
import matplotlib.pyplot as plt

In [ ]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        prompt = str(self.data.loc[index, 'prompt'])
        label = int(self.data.loc[index, 'label'])

        encoding = self.tokenizer.encode_plus(
            prompt,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = model.to(device)
loss_fn = torch.nn.CrossEntropyLoss().to(device)
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model.train()
    total_loss = 0
    correct_predictions = 0
    all_labels = []
    all_preds = []

    for data in tqdm(data_loader, desc="Training", leave=False):
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        labels = data['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        logits = outputs.logits
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='binary')
    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader), f1
def eval_model(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    all_labels = []
    all_probs = []
    all_preds = []

    with torch.no_grad():
        for data in tqdm(data_loader, desc="Evaluating", leave=False):
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_mask'].to(device)
            labels = data['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[:, 1]  # Binary
            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='binary')
    cm = confusion_matrix(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader), f1, cm, auc


In [ ]:
EPOCHS = 3
train_df = pd.read_csv('/content/training_data_binary.csv') #Insert Path to data

In [ ]:
train_df.rename(columns={'jailbreak': 'label'}, inplace=True)
val_df.rename(columns={'jailbreak': 'label'}, inplace=True)
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
train_dataset = TextDataset(train_df, tokenizer, max_len=128)
val_dataset = TextDataset(val_df, tokenizer, max_len=128)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16)
optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
total_steps = len(train_loader) * 5
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
def save_checkpoint(epoch, model, optimizer, scheduler, loss, file_path):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, file_path)
    print(f"Checkpoint saved at epoch {epoch}")

In [ ]:
def load_checkpoint(file_path, model, optimizer, scheduler):
    if os.path.exists(file_path):
        checkpoint = torch.load(file_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        epoch = checkpoint['epoch']
        loss = checkpoint['loss']
        print(f"Checkpoint loaded from {epoch} epoch")
        return epoch, loss
    else:
        print(f"No checkpoint found, starting from epoch 1.")
        return 0, None

In [ ]:
try:
    results = pd.read_csv('/content/results.csv') #Path to results file if exists
except FileNotFoundError:
    results = pd.DataFrame(columns=['Run', 'Best_Train_Accuracy', 'Best_Train_F1',
                                    'Best_Train_Loss', 'Best_Val_Accuracy', 'Best_Val_F1',
                                    'Best_Val_Loss', 'Best_Val_AUC'])

start_run = len(results) + 1

for run in range(start_run, 31):
    print(f"\nStarting Run {run}/30")
    train_sample = train_df.sample(frac=0.8, random_state=run).reset_index(drop=True)
    val_sample = train_df.drop(train_sample.index).reset_index(drop=True)
    train_dataset = TextDataset(train_sample, tokenizer, max_len=128)
    val_dataset = TextDataset(val_sample, tokenizer, max_len=128)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16)
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * EPOCHS)
    best_val_metrics = {
        'accuracy': 0,
        'f1': 0,
        'loss': float('inf'),
        'auc': 0
    }
    for epoch in range(EPOCHS):
        print(f'Epoch {epoch + 1}/{EPOCHS} for Run {run}')

        train_acc, train_loss, train_f1 = train_epoch(model, train_loader, loss_fn, optimizer, device, scheduler)
        val_acc, val_loss, val_f1, val_cm, val_auc = eval_model(model, val_loader, loss_fn, device)
        if val_acc > best_val_metrics['accuracy']:
            best_val_metrics.update({
                'accuracy': val_acc,
                'f1': val_f1,
                'loss': val_loss,
                'auc': val_auc
            })
    new_row = pd.DataFrame({
        'Run': [run],
        'Best_Train_Accuracy': [train_acc.item()],
        'Best_Train_F1': [train_f1],
        'Best_Train_Loss': [train_loss],
        'Best_Val_Accuracy': [best_val_metrics['accuracy'].item()],
        'Best_Val_F1': [best_val_metrics['f1']],
        'Best_Val_Loss': [best_val_metrics['loss']],
        'Best_Val_AUC': [best_val_metrics['auc']]
    })
    results = pd.concat([results, new_row], ignore_index=True)
    results.to_csv('results.csv', index=False)

In [ ]:
# Plots
plt.figure(figsize=(12, 8))

# Accuracy
plt.subplot(2, 2, 1)
plt.plot(results['Run'], results['Best_Train_Accuracy'], label='Train Accuracy', marker='o')
plt.plot(results['Run'], results['Best_Val_Accuracy'], label='Validation Accuracy', marker='o')
plt.xlabel('Run')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Best Accuracy per Run')

# F1 Score
plt.subplot(2, 2, 2)
plt.plot(results['Run'], results['Best_Train_F1'], label='Train F1 Score', marker='o')
plt.plot(results['Run'], results['Best_Val_F1'], label='Validation F1 Score', marker='o')
plt.xlabel('Run')
plt.ylabel('F1 Score')
plt.legend()
plt.title('Best F1 Score per Run')

# Loss
plt.subplot(2, 2, 3)
plt.plot(results['Run'], results['Best_Train_Loss'], label='Train Loss', marker='o')
plt.plot(results['Run'], results['Best_Val_Loss'], label='Validation Loss', marker='o')
plt.xlabel('Run')
plt.ylabel('Loss')
plt.legend()
plt.title('Best Loss per Run')

# AUC
plt.subplot(2, 2, 4)
plt.plot(results['Run'], results['Best_Val_AUC'], label='Validation AUC', marker='o')
plt.xlabel('Run')
plt.ylabel('AUC')
plt.legend()
plt.title('Best AUC per Run')

plt.tight_layout()
plt.show()